In [1]:
import uniport as up
import scanpy as sc
import torch
import scanpy as sc
import pandas as pd
import numpy as np
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import scanpy as sc
key = 'subclass.v4'

In [ ]:
T_names = [ "T996", "T928", "T906", "T758", "T996", "T756", "T993", "T989"]

# 单细胞数据文件路径
# 循环处理每个 profile
for profile in T_names:

    scadata = sc.read_h5ad('/data/work/Layer division/00_Data/snRNA_downsample_1000.h5ad')# preprocessed data with data2 specific HVG, as reference data
    stadata = sc.read_h5ad('/data/work/Layer division/00_Data/'+profile+'_Spatial-id.h5ad')  # preprocessed data with data1 specific HVG

    sc.pp.subsample(scadata, n_obs=4000)
    sc.pp.normalize_total(scadata)
    sc.pp.log1p(scadata)
    sc.pp.highly_variable_genes(scadata, n_top_genes=6000, inplace=False, subset=True)

    stadata.obs['X'] = stadata.obs['coor_x']
    stadata.obs['Y'] = stadata.obs['coor_y']
    ###数据加工
    # stadata.obs['cell_type'] = stadata.obs['leiden']
    stadata.obs['domain_id'] = 0
    stadata.obs['domain_id'] = stadata.obs['domain_id'].astype('category')
    stadata.obs['source'] = 'Spatial'

    scadata.obs['cell_type'] = scadata.obs[key]
    scadata.obs['domain_id'] = 1
    scadata.obs['domain_id'] = scadata.obs['domain_id'].astype('category')
    scadata.obs['source'] = 'RNA'


    # preprocess to get torch datasets and dataloaders
    from spann.preprocess import *

    adata_cm, adata_spa, adata_rna = anndata_preprocess(stadata, scadata,spatial_labels=False ,highly_variable=1000)
    source_sp_ds,target_sp_ds,source_cm_dl,target_cm_dl,test_source_cm_dl,test_target_cm_dl = generate_dataloaders(adata_cm,adata_spa,adata_rna)

    # contruct SPANN model
    from spann.model import *

    enc,dec,x_dim,z_dim = generate_ae_params(adata_cm, adata_spa, adata_rna)
    spann = SPANN_model(x_dim, z_dim, enc, dec, class_num=len(adata_rna.obs['cell_type'].unique()), device=device)

    # train SPANN model without validation

    source_sp_ds,target_sp_ds,source_cm_dl,target_cm_dl,\
    test_source_cm_dl,test_target_cm_dl = generate_dataloaders(adata_cm,adata_spa,adata_rna,batch_size=256)
    adata_source, adata_target, threshold_test = spann.train(source_cm_dl,target_cm_dl,source_sp_ds,target_sp_ds,adata_spa.obs[["X","Y"]],
                                                             test_source_cm_dl, test_target_cm_dl,np.array(adata_rna.obs['labels']),np.unique(adata_rna.obs['cell_type']),
                                                             lr=2e-4,resolution=0.5,lambda_spa=0.001,lambda_cd=0.001,lambda_nb=10,
                                                             maxiter=5000,miditer1=2000,miditer2=4000,miditer3=4000)

    # umap visualizations
    # umap plot colored by source and cell types

    adata_total = sc.AnnData.concatenate(adata_source, adata_target)
    sc.pp.neighbors(adata_total, use_rep="X")
    sc.tl.umap(adata_total)
    sc.pl.umap(adata_total, color="source", show=True)
    sc.pl.umap(adata_total[adata_total.obs['source']=='scRNA'], color="cell_type", show=True)
    sc.pl.umap(adata_total[adata_total.obs['source']=='Spatial'], color="pred_cls", show=True)

    adata_target.obs['XY'] = adata_target.obs['X'].astype(str) + '_' + adata_target.obs['Y'].astype(str)
    data = {
        'index': adata_target.obs['XY'],
        'pred_cls': adata_target.obs['pred_cls']
    }
    df = pd.DataFrame(data)
    df.to_csv('/data/work/Layer division/01_Result/Spann_'+profile+'.cellbin_pret.csv')


    ####Unipot

    stadata.obs['source'] = 'SPOT'
    scadata.obs['source'] = 'RNA'
    adata_cm = stadata.concatenate(scadata, join='inner', batch_key='domain_id')


    sc.pp.normalize_total(adata_cm)
    sc.pp.log1p(adata_cm)
    sc.pp.highly_variable_genes(adata_cm, batch_key='domain_id', inplace=False, subset=True)
    up.batch_scale(adata_cm)

    adata, OT = up.Run(adatas=[stadata,scadata], adata_cm=adata_cm, num_workers=16,save_OT=True)
    name_idx = adata_cm[adata_cm.obs['source']=='SPOT'].obs_names
    name_col = adata_cm[adata_cm.obs['source']=='RNA'].obs[key]
    df = pd.DataFrame(OT[0], index=name_idx, columns=name_col)

    max_values = df.max(axis=1)
    # Get the column name corresponding to the maximum value in each row
    max_column_names = df.idxmax(axis=1)

    # Concatenate the original row names with the corresponding column name of the maximum value
    new_index =  max_column_names

    new_index.to_csv('/data/work/Layer division/01_Result/Uniport_'+profile+'.cellbin_pret.csv')



... storing 'orig.ident' as categorical
... storing 'source' as categorical


rna_labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24]
#Iter 100: Reconstruction loss: 2.191981, KL loss: 90.660110, CLS loss: 3.263066, Spatial loss: 0.000000, CCD loss: 0.000000, Neighbor loss: 0.000000
#Iter 200: Reconstruction loss: 2.178203, KL loss: 87.615158, CLS loss: 2.974957, Spatial loss: 0.000000, CCD loss: 0.000000, Neighbor loss: 0.000000


In [ ]:
1